# HMSTI 509 — Module 10 Lesson
## Ship Your Agent

**The whole idea:** you have seen loop + tools + RAG. Today we expand the agent by introducing more tools.

Agenda:

1. **Recap (Week 7):** the `Tool` class.
2. **Recap (Week 7):** the registry and `dispatch()`.
3. **Recap (Week 8):** the agent loop and its cap.
4. **Many tools, one loop:** put the registry inside the loop.
5. See how the model picks a tool: from its **description**.
6. Build the first tool of your own.
7. **Harden** the agent: discuss where things can go wrong and how to reduce the risk.

Sections 1–3 are code you have already run, copied unchanged from Weeks 7 and 8. Section 4 puts
those pieces together.


## 0. Setup — point Python at your `rag-starter` repo

The cell below finds `rag-starter/` automatically and points the agent at your local **Ollama**
model from Week 8 (`qwen3.5:2b` by default). Start the Ollama app before you run it.

Some demos script the model's moves on purpose (the multi-tool turns in section 4 and the tool-choice
cases in sections 5–6), so every run shows the same behavior. The live demo in section 4 talks to
your Ollama model. Section 7 is a discussion with no code to run.

> No Ollama, or the model isn't pulled? The cell falls back to the built-in offline stand-in and
> tells you why. Every cell still runs; you just won't see a real model's choices.


In [1]:
# GIVEN — run as-is.
import json
import os
import sys
import urllib.request
from pathlib import Path

# Walk up from this notebook until we find the rag-starter package, then put it on sys.path.
here = Path.cwd() # current working directory
for parent in [here, *here.parents]:
    REPO = parent / "rag-starter"
    if (REPO / "ribot").is_dir(): # check for the ribot package inside rag-starter
        sys.path.insert(0, str(REPO))
        print("Found rag-starter at:", REPO)
        break
else:
    print("Could not find rag-starter/ — open this notebook from inside the course repo.")

# Use your local Ollama model if it is running and pulled; otherwise fall back to the offline stand-in.
from ribot.config import get_settings

settings = get_settings()
try:
    with urllib.request.urlopen(f"{settings.ollama_host}/api/tags", timeout=3) as response:
        pulled = [model["name"] for model in json.load(response)["models"]]
except OSError:
    pulled = None
if pulled and settings.ollama_model in pulled:
    os.environ["RIBOT_LLM"] = "ollama"
    print("Model: Ollama", settings.ollama_model)
else:
    os.environ["RIBOT_LLM"] = ""
    why = "Ollama is not running" if pulled is None else f"run `ollama pull {settings.ollama_model}`"
    print(f"Model: offline stand-in ({why}). Every cell still runs.")

Found rag-starter at: /Users/luyaoniu/Desktop/Teaching-UW/HMSTI 509/2026/rag-starter
Model: Ollama qwen3.5:2b


The next cell builds the index over the sample documents and defines helpers for the demos:
scripted stand-in models, a trace printer, and a function that reports which tool a model picked.
Run the cell as-is; you don't need to read it.


In [2]:
# GIVEN — run as-is. Helpers for the demos.
import tempfile
from dataclasses import replace
from datetime import date

from ribot.agent_loop import run_harness_turn
from ribot.build_index import build_index
from ribot.embeddings import _tokens
from ribot.harness import make_tools
from ribot.my_tools import MY_TOOLS
from ribot.toolbox import deny_all, make_assistant_tools

docs = build_index(quiet=True)       # the sample documents every demo searches
retriever = docs                     # the same index, under its Week 8 name
SCRATCH = Path(tempfile.mkdtemp())   # anything the demos write lands here, never in the repo


def scripted(*replies):
    """A stand-in TEXT model: says `replies` in order."""
    queue = iter(replies)
    return lambda messages: next(queue, "(the script has nothing more to say)")



def toolbox_with(*extra, replace_tools=()):
    """The terminal assistant's toolbox (+ MY_TOOLS + extra), actions declined; swap in replace_tools by name."""
    swaps = {tool.name: tool for tool in replace_tools}
    menu = make_assistant_tools(docs, confirm=deny_all, extra=[*MY_TOOLS, *extra],
                                memory_path=SCRATCH / "assistant_memory.md", drafts_dir=SCRATCH / "drafts")
    return [swaps.get(tool.name, tool) for tool in menu]


IGNORE = {"what", "which", "who", "when", "where", "why", "use", "user", "asks"}


def stems(text):
    return {w[:5] for w in _tokens(text.replace("_", " ")) if len(w) > 2 and w not in IGNORE}


def description_reader(messages, declarations):
    """Stand-in model: call the tool whose name + description shares the most words with the request."""
    if any(isinstance(m.get("content"), list) for m in messages):
        return {"role": "assistant", "stop_reason": "end_turn",
                "content": [{"type": "text", "text": "(answer written from the tool result)"}]}
    question = messages[-1]["content"]
    overlap = [len(stems(question) & stems(d["name"] + " " + d["description"])) for d in declarations]
    if max(overlap) > 0:
        chosen = declarations[overlap.index(max(overlap))]
    else:
        chosen = next((d for d in declarations if d["name"] == "search_docs"), declarations[0])
    args = {field: question for field in chosen["input_schema"].get("required", [])}
    return {"role": "assistant", "stop_reason": "tool_use",
            "content": [{"type": "tool_use", "id": "toolu_standin", "name": chosen["name"], "input": args}]}


def choice_report(tools, cases, chat_fn=description_reader):
    passed = 0
    for question, expected in cases:
        turn = run_harness_turn(question, tools, chat_fn=chat_fn)
        called = turn.steps[0].name if turn.steps else "none"
        passed += called == expected
        print(f"  [{'PASS' if called == expected else 'FAIL'}] {question[:52]!r:56} expected {expected:<13} called {called}")
    print(f"  {passed}/{len(cases)} picked the right tool first")
    return passed


tools = [*make_tools(docs), *MY_TOOLS]          # search_docs, list_sources, current_date
print("helpers ready; tools:", [tool.name for tool in tools])


helpers ready; tools: ['search_docs', 'list_sources', 'current_date']


## 1. Recap (Week 7): the `Tool` class

A tool bundles what the model needs to **choose** it (`name`, `description`, `input_schema`) with
the function our code **runs** (`run`). The model sees the first three and never sees `run`.

`call()` never raises errors. Errors are handled by `ToolResult` with `is_error=True`. The loop hands that result to the model, which can fix its call and try again, instead of the whole conversation crashing.


In [3]:
from dataclasses import dataclass
from typing import Callable

# JSON Schema type names -> the Python types that satisfy them
JSON_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool}


@dataclass(frozen=True)
class ToolResult:
    """What a tool hands back to the model: text, plus a flag when something went wrong."""
    content: str
    is_error: bool = False


@dataclass(frozen=True)
class Tool:
    """One tool = what the model sees (name, description, schema) + the function we run."""
    name: str
    description: str
    input_schema: dict
    run: Callable[..., str]       # any function that returns a string
    read_only: bool = True        # False = it changes something (saves, sends, deletes)

    def declaration(self) -> dict:
        """The part the model sees, in the Anthropic / MCP shape."""
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": self.input_schema,
        }

    def to_openai(self) -> dict:
        """The same declaration in OpenAI's envelope."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.input_schema,
            },
        }

    def check_args(self, args: dict) -> str:
        """Return '' if args fit the schema, otherwise what is wrong."""
        properties = self.input_schema.get("properties", {})
        for field in self.input_schema.get("required", []):
            if field not in args:
                return f"missing required argument {field!r}"
        for field, value in args.items():
            if field not in properties:
                return f"unknown argument {field!r}"
            expected = JSON_TYPES.get(properties[field].get("type"))
            if expected and not isinstance(value, expected):
                return f"{field!r} should be a {properties[field]['type']}"
        return ""

    def call(self, args: dict) -> ToolResult:
        """Check the arguments, run the function, and turn any failure into an error result."""
        problem = self.check_args(args)
        if problem:
            return ToolResult(f"Bad arguments for {self.name}: {problem}", is_error=True)
        try:
            output = self.run(**args)
        except Exception as err:          # reported to the model, never hidden
            return ToolResult(f"{self.name} failed: {err}", is_error=True)
        if not isinstance(output, str):
            return ToolResult(f"{self.name} returned {type(output).__name__}, not text", is_error=True)
        return ToolResult(output)

print("Tool and ToolResult defined")

Tool and ToolResult defined


## 2. Recap (Week 7): the registry and `dispatch()`

A **registry** is a dict from tool name to tool. The model sends back a request shaped like `{"name": ..., "input": {...}}`. 

`dispatch()` looks the name up in the registry and calls that tool. 

In [4]:
NOTES = [
    "To reset your password, use the 'Forgot password' link on the sign-in page.",
    "The Team plan costs 8 dollars per user per month.",
    "Support is open Monday to Friday, 9am to 5pm Pacific.",
]
saved_notes: list[str] = []


def search_notes(query: str, k: int = 1) -> str:
    """Return the k notes sharing the most words with the query."""
    words = set(query.lower().split())
    scored = [(len(words & set(note.lower().split())), note) for note in NOTES]
    scored.sort(reverse=True)
    return "\n".join(note for _score, note in scored[:k])


def save_note(text: str) -> str:
    saved_notes.append(text)
    return f"Saved. You now have {len(saved_notes)} note(s)."


search_tool = Tool(
    name="search_notes",
    description=(
        "Search the team's help notes. Use it for questions about passwords, "
        "pricing, or support hours. Returns the k best-matching notes."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "What to look up"},
            "k": {"type": "integer", "description": "How many notes to return"},
        },
        "required": ["query"],
    },
    run=search_notes,
)

save_tool = Tool(
    name="save_note",
    description="Save a short note for the user. Use it only when the user asks you to remember something.",
    input_schema={
        "type": "object",
        "properties": {"text": {"type": "string", "description": "The note to save"}},
        "required": ["text"],
    },
    run=save_note,
    read_only=False,              # it changes something
)

registry = {tool.name: tool for tool in [search_tool, save_tool]}


def dispatch(request: dict) -> ToolResult:
    """Run the tool a model asked for. `request` looks like {'name': ..., 'input': {...}}."""
    tool = registry.get(request.get("name"))
    if tool is None:
        return ToolResult(
            f"Unknown tool {request.get('name')!r}. Available: {sorted(registry)}", is_error=True
        )
    return tool.call(request.get("input", {}))


# Pretend the model sent these three requests:
for request in [
    {"name": "search_notes", "input": {"query": "how much is the team plan"}},
    {"name": "search_notes", "input": {"k": 2}},              # forgot the query
    {"name": "send_email", "input": {"to": "my boss"}},       # no such tool
]:
    print(request["name"], "->", dispatch(request))

search_notes -> ToolResult(content='The Team plan costs 8 dollars per user per month.', is_error=False)
search_notes -> ToolResult(content="Bad arguments for search_notes: missing required argument 'query'", is_error=True)
send_email -> ToolResult(content="Unknown tool 'send_email'. Available: ['save_note', 'search_notes']", is_error=True)


What the model actually receives is each tool's **declaration**: its entry on the menu. `run` is
in neither format:


In [13]:
# What the model actually receives: the same tool in two envelopes. `run` is in neither.
print(json.dumps(search_tool.declaration(), indent=2))


{
  "name": "search_notes",
  "description": "Search the team's help notes. Use it for questions about passwords, pricing, or support hours. Returns the k best-matching notes.",
  "input_schema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "What to look up"
      },
      "k": {
        "type": "integer",
        "description": "How many notes to return"
      }
    },
    "required": [
      "query"
    ]
  }
}


## 3. Recap (Week 8): the agent loop

The Week 8 agent had **one** tool, document search. The model asked for it with a line starting `SEARCH:`. Our code ran the retriever, appended the results to `messages`, and went around again.

A reply without `SEARCH:` is the final answer.

`retriever` is the index the setup cell built. The first cell below switches to the offline stand-in, which plays the `SEARCH:` protocol exactly as it did in Week 8. 



In [6]:
# GIVEN — run the Week 8 recap on the offline stand-in.
MODEL_CHOICE = os.environ["RIBOT_LLM"]   # what section 0 picked
os.environ["RIBOT_LLM"] = ""


In [7]:
from ribot.llm import chat
from ribot.agent import AGENT_SYSTEM

def run_agent_traced(question, retriever, max_iters=3):
    """Same loop as ribot.agent.run_agent, but it PRINTS each step so we can trace it."""
    messages = [
        {"role": "system", "content": AGENT_SYSTEM},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_iters + 1):
        reply = chat(messages)
        if reply.strip().startswith("SEARCH:"):
            query = reply.split("SEARCH:", 1)[1].strip()
            print(f"step {step}: model asked to SEARCH -> {query!r}")
            found = retriever.search_text(query, k=3)
            print(f"step {step}: ran the tool, fed back {len(found)} chars of results")
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": f"SEARCH RESULTS: {found}"})
            continue
        print(f"step {step}: model gave a FINAL answer (no SEARCH) -> done")
        return reply
    return "I couldn't finish within the step limit."

print("FINAL:", run_agent_traced("How do I reset my password?", retriever))

step 1: model asked to SEARCH -> 'How do I reset my password?'
step 1: ran the tool, fed back 910 chars of results
step 2: model gave a FINAL answer (no SEARCH) -> done
FINAL: Based on the documents: # Acme Cloud — Help Center (synthetic sample data)

## Resetting your password
If you forgot your password, click "Forgot password" on the sign-in page and enter the
email on your account. You will receive a reset link that is valid for 30


In [10]:
# GIVEN — back to the model section 0 picked.
os.environ["RIBOT_LLM"] = MODEL_CHOICE
print("Model:", MODEL_CHOICE or "offline stand-in")


Model: ollama


## 4. Many tools in loop

Compare section 2 with section 3. The Week 8 loop runs exactly one tool, and that tool is written
into the loop itself:

```python
if reply.strip().startswith("SEARCH:"):
    ...
    found = retriever.search_text(query, k=3)   #  hard-wired one tool
```

To add a second tool this way, you add an `elif`, then another for the third tool, and so on.
The loop grows with every tool, and each new branch is a new place for a bug.

Week 7 already gave us the alternative: `dispatch(request)` runs **any** tool in the registry by name. So we change the Week 8 loop in two places:

1. **The model names the tool.** The model replies with one line that
   holds the same request dict `dispatch()` takes:
   `TOOL: {"name": "search_docs", "input": {"query": "password reset"}}`
2. **The loop calls `dispatch()`** in place of the hard-wired search.



### 4a. Grow the registry


In [11]:
def search_docs(query: str) -> str:
    """The Week 8 tool: the 3 best-matching chunks from the document index."""
    return retriever.search_text(query, k=3)


def current_date() -> str:
    return date.today().isoformat() # YYYY-MM-DD format


docs_tool = Tool(
    name="search_docs",
    description=(
        "Search the user's documents: help center, HR policy, product FAQ. Use it for any "
        "question those documents could answer. Returns the 3 best-matching passages."
    ),
    input_schema={
        "type": "object",
        "properties": {"query": {"type": "string", "description": "What to look up"}},
        "required": ["query"],
    },
    run=search_docs,
)

date_tool = Tool(
    name="current_date",
    description="Get today's date as YYYY-MM-DD. Use it when the user asks what today's date is.",
    input_schema={"type": "object", 
                  "properties": {}},
    run=current_date,
)

# The registry from section 2, now with four tools. dispatch() looks up this name, so it sees all four.
registry = {tool.name: tool for tool in [docs_tool, search_tool, save_tool, date_tool]}
print(sorted(registry))


['current_date', 'save_note', 'search_docs', 'search_notes']


### 4b. Put the menu in the system prompt

In Week 8, `AGENT_SYSTEM` described the one tool in words. With several tools, we build the
system prompt from the registry instead: every tool's declaration (section 2) goes in. The menu
the model reads then always matches the tools our code can run.


In [ ]:
def tools_prompt(registry):
    """System prompt: the menu (every tool's declaration) plus the TOOL: protocol."""
    menu = [tool.declaration() for tool in registry.values()]
    return (
        "You are a helpful assistant with these tools:\n"
        + json.dumps(menu, indent=2)
        + "\n\nTo use a tool, reply with ONE line and nothing else:\n"
        'TOOL: {"name": "<tool name>", "input": {<arguments>}}\n'
        "After a line beginning with 'TOOL RESULT' or 'TOOL ERROR', use it to answer the user, "
        "or call another tool. When you can answer, reply in plain text."
    )


print(tools_prompt(registry)[:700], "\n...")


### 4c. The loop

Read it next to `run_agent_traced` from section 3. Most lines are the same. The `if SEARCH:` block
now does three things: read the request, `dispatch()` it, and feed the result back.

- `parse_request` returns `None` when the `TOOL:` line isn't valid JSON. The loop sends the model an
  error message for it, the same way `dispatch()` handles an unknown tool name.
- The `chat_fn` parameter lets us pass a scripted stand-in, like `AlwaysSearchLLM` in section 3. By
  default it is the same `chat()`.

The loop never names a tool. Add a fifth tool to the registry and this code doesn't change.


In [ ]:
from ribot.llm import chat


def parse_request(reply):
    """'TOOL: {"name": ..., "input": {...}}' -> the request dict, or None if it can't be read."""
    try:
        request = json.loads(reply.split("TOOL:", 1)[1])
    except json.JSONDecodeError:
        return None
    if not isinstance(request, dict) or not isinstance(request.get("input", {}), dict):
        return None
    return request


def run_agent_multi(question, chat_fn=chat, max_iters=4):
    """The Week 8 loop, with dispatch() in place of the one hard-wired search."""
    messages = [
        {"role": "system", "content": tools_prompt(registry)},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_iters + 1):
        reply = chat_fn(messages)
        if reply.strip().startswith("TOOL:"):
            request = parse_request(reply)
            if request is None:
                result = ToolResult(
                    'Could not read that TOOL line. Reply with ONE line and nothing else: '
                    'TOOL: {"name": ..., "input": {...}}', is_error=True
                )
            else:
                result = dispatch(request)                  # was: retriever.search_text(query, k=3)
            label = "TOOL ERROR" if result.is_error else "TOOL RESULT"
            print(f"step {step}: model asked -> {reply.strip()[:75]!r}")
            print(f"        {label}: {result.content[:75]!r}")
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": f"{label}: {result.content}"})
            continue
        print(f"step {step}: model gave a FINAL answer (no TOOL) -> done")
        return reply
    return "I couldn't finish within the step limit."


### 4d. Two tools in one turn

The model's moves below are scripted with `scripted(...)`, so every run prints the same trace. We
wrote the model's decisions; what we're watching is the loop.

**Predict before you run:** how many times does the loop call `dispatch()`, and how many times does
it call the model?


In [ ]:
answer = run_agent_multi(
    "What's today's date, and how much is the Team plan?",
    chat_fn=scripted(
        'TOOL: {"name": "current_date", "input": {}}',
        'TOOL: {"name": "search_notes", "input": {"query": "how much is the team plan"}}',
        f"Today is {date.today().isoformat()}, and the Team plan costs 8 dollars per user per month.",
    ),
)
print("FINAL:", answer)


Two tools, three model calls: one per tool request, plus one for the final answer. Every step is
a model call, so with an API model every step costs money. That is why the cap stays.

### 4e. When the model gets it wrong

The same mistakes as in Week 7's `dispatch()` demo, now inside the loop: a tool that doesn't exist,
a missing argument, and a line that isn't JSON. Each one becomes a message the model can read, and
the loop keeps going.


In [ ]:
answer = run_agent_multi(
    "Is it raining in Seattle?",
    chat_fn=scripted(
        'TOOL: {"name": "weather", "input": {"city": "Seattle"}}',   # no such tool
        'TOOL: {"name": "search_docs", "input": {}}',                # forgot the query
        "TOOL: search_docs | rain",                                  # not JSON
        "I don't have a weather tool, so I can't check the weather in Seattle.",
    ),
)
print("FINAL:", answer)


The unknown-tool error lists the real tools, so the model can tell the user it has no weather
tool. An `elif` chain has no branch for `weather`, so it would have crashed or quietly reused an old
result.

### 4f. A trap: two tools with the same name

Tools come from several places: the starter, your `my_tools.py`, a teammate's code, later an MCP
server. Two of them can share a name, and a dict keeps only the **last** value for each key.


### 4g. Now with your Ollama model

So far we scripted the model's moves. The cell below runs the same loop with your Ollama model.
Small models often break a text protocol: they add words around the `TOOL:` line, write invalid
JSON, or make up a tool name. Look for steps where an error message from 4e helped the model
correct itself, and for runs the cap had to stop.

While this lesson was being prepared, `qwen3.5:2b` answered *What's today's date?* with a correct
`TOOL:` line, then wrote its own made-up `TOOL RESULT` on the next line. `parse_request` rejected
the whole reply, and the model then answered with a date it made up, about a year off. The loop
did its job. The weak point is the text protocol, which is why production APIs use structured
tool calls (next).


In [ ]:
from ribot.agent_eval import using_scripted_model

if using_scripted_model():
    print("Offline: skipped. Start Ollama, re-run the section 0 cell, then this one.")
else:
    for question in ["What's today's date?", "How do I reset my password?"]:
        print(f"===== {question}")
        print("FINAL:", run_agent_multi(question))
        print()


### From a text protocol to the production format

Parsing a `TOOL:` line by hand is fragile, and 4g may have shown you why. Model APIs (Anthropic,
OpenAI, and Ollama's tool calling) return a tool request as a **structured block**, so the loop
has no text to parse. Your repo runs that version of the loop in `run_harness_turn()`
(`ribot/agent_loop.py`), and the rest of this notebook uses it. Each part matches something you
just wrote:

| `run_agent_multi()` (this section) | `run_harness_turn()` (`ribot/agent_loop.py`) |
|---|---|
| `for step in range(1, max_iters + 1)` | the same cap |
| `tools_prompt(registry)`: the menu inside the system prompt | the declarations, sent to the API as a separate `tools` list |
| `parse_request(reply)` | `_first_tool_use(reply)`: read the `tool_use` block |
| `dispatch(request)` | `_run_tool(registry, block)` |
| a `TOOL RESULT: ...` message | a `tool_result` block, tied to its request by an `id` |
| `return reply` | `return AgentTurn(answer, steps, hit_cap)` |

`AgentTurn.steps` lists every tool call the turn made. `hit_cap=True` means the cap stopped the loop.


## 5. (Optional) Tool choice: descriptions decide

Our code decides whether a requested call runs. The model decides which tool to request, and the only
evidence it has is the menu: names and descriptions.

Here `description_reader` stands in for the model, so the report comes out the same every run. It
picks the tool whose name and description share the most words with the question. A real model
reads meaning, and it works from the same evidence; you will run your Ollama model on the same
cases at the end of this section.

`CHOICE_CASES` pairs each question with the tool that should answer it. `choice_report` runs each
question and checks which tool was called first:

In [15]:
CHOICE_CASES = [
    ("How do I reset my password?", "search_docs"),
    ("How much does the Business plan cost?", "search_docs"),
    ("Which documents do you have?", "list_sources"),
    ("What's today's date?", "current_date"),
    ("Please remember that I prefer short bullet-point answers.", "remember"),
    ("Save a draft reply to the vendor saying we ship Friday.", "save_draft"),
]
choice_report(toolbox_with(), CHOICE_CASES)

  [PASS] 'How do I reset my password?'                            expected search_docs   called search_docs
  [PASS] 'How much does the Business plan cost?'                  expected search_docs   called search_docs
  [PASS] 'Which documents do you have?'                           expected list_sources  called list_sources
  [PASS] "What's today's date?"                                   expected current_date  called current_date
  [PASS] 'Please remember that I prefer short bullet-point ans'   expected remember      called remember
  [PASS] 'Save a draft reply to the vendor saying we ship Frid'   expected save_draft    called save_draft
  6/6 picked the right tool first


6

`CHOICE_CASES` can be used for evaluation, or fine-tune the model for tool calling

## 6. (Optional) Build your first tool


In [13]:
def days_until(target: str) -> str:
    """Whole days from today until `target` (YYYY-MM-DD)."""
    try:
        when = date.fromisoformat(target.strip())
    except ValueError:
        raise ValueError(f"could not read {target!r} as a date; send it as YYYY-MM-DD, e.g. 2026-12-31")
    days = (when - date.today()).days
    if days < 0:
        return f"{when.isoformat()} was {-days} days ago."
    return f"{days} days from today until {when.isoformat()}."


DAYS_UNTIL = Tool(
    name="days_until",
    description=("Count the days from today until a date. Use it when the user asks how long until a "
                 "deadline, event or due date. For today's date itself, use current_date."),
    input_schema={"type": "object",
                  "properties": {"target": {"type": "string", "description": "The date, as YYYY-MM-DD"}},
                  "required": ["target"]},
    run=days_until,
)

print(DAYS_UNTIL.call({"target": "2026-12-31"}))
print(DAYS_UNTIL.call({"target": "next Friday"}))
print(DAYS_UNTIL.call({}))

ToolResult(content='36 days from today until 2026-12-31.', is_error=False)
ToolResult(content="days_until failed: could not read 'next Friday' as a date; send it as YYYY-MM-DD, e.g. 2026-12-31", is_error=True)
ToolResult(content="Bad arguments for days_until: missing required argument 'target'", is_error=True)


In [16]:
DAYS_CASES = [("How many days until the 2026-12-31 deadline?", "days_until")]
choice_report(toolbox_with(DAYS_UNTIL), CHOICE_CASES + DAYS_CASES)
print("\ncurrent_date says:", MY_TOOLS[0].description)

  [PASS] 'How do I reset my password?'                            expected search_docs   called search_docs
  [PASS] 'How much does the Business plan cost?'                  expected search_docs   called search_docs
  [PASS] 'Which documents do you have?'                           expected list_sources  called list_sources
  [PASS] "What's today's date?"                                   expected current_date  called current_date
  [PASS] 'Please remember that I prefer short bullet-point ans'   expected remember      called remember
  [PASS] 'Save a draft reply to the vendor saying we ship Frid'   expected save_draft    called save_draft
  [FAIL] 'How many days until the 2026-12-31 deadline?'           expected days_until    called current_date
  6/7 picked the right tool first

current_date says: Get today's date (YYYY-MM-DD). Use it whenever the user asks about today, deadlines, or how many days until something.


It lost to an older tool. `current_date` was written when it was the only date tool, and its
description claims questions about *how many days until something*. Adding a tool means re-reading
the descriptions already on the menu. Narrow `current_date`'s description and run the cases again:

In [ ]:
CURRENT_DATE_NARROW = replace(MY_TOOLS[0], description=(
    "Get today's date (YYYY-MM-DD). Use it when the user asks what today's date is."))
choice_report(toolbox_with(DAYS_UNTIL, replace_tools=[CURRENT_DATE_NARROW]), CHOICE_CASES + DAYS_CASES)

### Sketch your tool

Questions to start from:

- What can't a model do on its own in your job? Look something up in a list you keep, compute
  something exact, check a value against a rule, format text to a house template.
- Your description needs a *Use it when …* sentence. Which tool already on the menu could claim the
  same questions?

## 7. (Optional) Harden: where things go wrong

**Hardening** means anticipating failures and limiting their consequences. A model can choose the
wrong tool, a service can fail, and retrieved text can contain misleading facts or instructions.
A successful demo does not show that these cases are handled. For each risk, ask: **How would we
notice, what should the agent do, and what should it refuse to do?**

### Bad tool requests and unreliable tools

| Where things may go wrong | Typical mitigation strategies |
|---|---|
| The model invents a tool name, omits an argument, sends the wrong type, or produces a malformed request. | Accept only registered tools; validate the request and arguments before execution. Return a clear error so the model can correct the request within a limited number of attempts. Structured tool calls reduce parsing errors, but still need validation. |
| Two tools share a name, or their descriptions claim the same questions. | Reject duplicate names when building the registry. Give each tool a distinct scope, and rerun behavior cases whenever a tool or description changes. |
| A tool crashes, a service is unavailable, or its output has an unexpected format. | Catch expected failures, validate returned data, and report a useful error without exposing secrets. Use timeouts and a small retry budget for temporary service failures; stop and explain when recovery fails. |
| The agent repeats a failed call, waits indefinitely, or spends too much. | Bound loop iterations, individual tool duration, and total time or cost. Keep a trace of calls and outcomes so repeated failures are visible. An iteration cap alone cannot stop a tool call that hangs. |

An error message gives the model a chance to recover; it does not guarantee recovery. For example,
a missing date can be corrected by asking the user, while an unavailable service may require ending
the turn with an explanation. Repeating the same call is not a recovery strategy.

### Unwanted actions and prompt injection

A retrieved help document might contain a useful password-reset answer followed by an instruction
to save private information in a draft. This is **prompt injection**: untrusted content tries to
direct the agent's behavior. With action tools, the consequence can extend beyond a wrong answer
to a changed file, an unwanted message, or persistent memory.

Typical mitigations work at several boundaries:

- Treat documents and tool results as evidence, rather than authority to issue new instructions.
  Clear prompts and source boundaries help the model recognize this distinction, but cannot
  guarantee that it will follow it.
- Give tools only the permissions, files, and destinations they need (**least privilege**).
  Read-only access can still expose private data, so limit what the agent can read as well.
- Put an **approval gate** in the execution path before actions. Show the exact action, destination,
  and content for review; decline by default. A refusal must prevent execution even if the model
  requests it again. Approval should apply to the reviewed action, not all future actions.
- If an action times out, check whether it already happened before retrying. Otherwise a retry
  could send the same message or create the same record twice.

The model's request and the program's permission to execute are separate decisions. A gate limits
unapproved actions; it does not make retrieved content trustworthy or guarantee a correct answer.

### Unsupported answers and weak retrieval

A search can return no useful passages, an outdated policy, or a plausible passage that does not
support the answer. The model may still sound confident or attach a citation that does not support
its claim.

Typical mitigations include keeping source documents current, checking retrieval on representative
questions, and requiring answers to cite supporting passages. A **score floor** can filter weak
matches, but a high similarity score is not proof that a passage is true or relevant. When evidence
is missing, conflicting, or insufficient, the agent should say what it cannot establish, ask a
clarifying question, or refer the user to a person who can verify it. Test unanswerable questions
as well as questions with known answers.

### Secrets, logs, and saved memory

API keys or private content can leak through a committed configuration file, an error message,
a trace, or saved memory. Keep secrets outside version control, restrict access to stored data,
and redact sensitive values from logs and tool results. Review what the agent saves and provide a
way to correct or delete it. If a key leaks, revoke or rotate it at the provider; deleting the file
from the current folder does not invalidate the exposed key.

**Discuss:** a document tells the agent to save private information, the save operation times out,
and the model proposes trying again. Which boundaries should prevent the first action? What would
you need to check before allowing a retry? What information belongs in the trace, and what should
be left out?


## Recap

The agent combines tools with descriptions, a registry, and a capped loop. Hardening adds checks
around that loop and limits the consequences when the model, a tool, or a source goes wrong.

- **Many tools, one loop:** the registry holds the tools, and the dispatch step runs the one the
  model names. Adding a tool does not require a new branch in the loop.
- **Tool choice needs evaluation:** distinct descriptions help the model choose. Reject duplicate
  names and rerun behavior cases when the menu changes.
- **Failures need bounded recovery:** validate requests and results, return useful errors, and
  limit retries, loop steps, and tool duration. The model may still need to stop or ask for help.
- **Actions need boundaries:** treat retrieved instructions as untrusted, restrict tool permissions,
  and require approval for the exact action. Check whether an action succeeded before retrying it.
- **Answers need evidence:** check that cited passages support the claim. A score floor filters weak
  matches; it cannot guarantee correctness. Acknowledge missing or conflicting evidence.
- **Stored data needs care:** keep secrets out of version control and logs, review saved memory,
  and revoke or rotate exposed keys.

**Where this goes next.** **MCP (Model Context Protocol)** lets servers expose tools for agents to
discover and call. Those tools still need distinct names, clear descriptions, validation, permission
boundaries, and evaluation. Agent frameworks add features around these same decisions.

**The assignment:** follow the assignment notebook for the two tools, registry, hardening checks,
documentation, reproducibility check, demo video, and 90-second AI-free explanation. The Week 10
experiment card is there too. Office hours stay open for 48 hours after the session.


## Vocabulary review

Use the examples to explain each term in your own words. For sections 1–6, connect each term to the code. For hardening, explain a failure
mode and a mitigation from section 7.

| Topic | Vocabulary | Meaning | Concrete example from this lesson |
|---|---|---|---|
| Tools | **Tool** | A function your code runs on the model's behalf, packaged with the name, description, and schema the model reads. | `docs_tool = Tool(name="search_docs", description=..., input_schema=..., run=search_docs)` |
| Tools | **Registry** | A dict from tool name to tool, so the loop can find any tool by the name the model asks for. | `registry = {tool.name: tool for tool in [docs_tool, search_tool, save_tool, date_tool]}` |
| Tools | **Dispatch step** | The part of the loop that looks up the requested tool and runs it, returning an error result for an unknown name. | `dispatch({"name": "weather", "input": {...}})` returns `Unknown tool 'weather'. Available: [...]`. |
| Tools | **Duplicate tool name** | Two tools with the same name; a plain dict keeps only the last one, silently. | If two entries are named `search_docs`, a dict keeps only the last; section 4f explains this risk. |
| Tools | **Menu / declaration** | What the model sees about each tool: name, description, and input schema. The `run` function stays in your code. | `tools_prompt(registry)` puts every `tool.declaration()` into the system prompt. |
| Tools | **Input schema** | A description of the arguments a tool accepts, which `call()` checks before running the tool. | `days_until` requires one string field, `target`, as YYYY-MM-DD. |
| Tools | **Error result (`ToolResult`, `is_error`)** | A failure returned to the model as a readable message, so the loop keeps going and the model can retry. | `DAYS_UNTIL.call({"target": "next Friday"})` returns an error that says to send YYYY-MM-DD. |
| Agent loop | **Text tool protocol** | The agreed line format a model uses to ask for a tool by name, with its arguments. | `TOOL: {"name": "search_docs", "input": {"query": "password reset"}}`, read by `parse_request()`. |
| Agent loop | **Iteration cap** | The maximum number of loop passes, so the loop always ends. | `run_agent_multi` uses `max_iters=4`; a separate timeout is needed for a hanging tool. |
| Production format | **`tool_use` block** | The structured request a model API sends when the model wants a tool; the job the `TOOL:` line did. | `_first_tool_use(reply)` in `run_harness_turn()` reads it. |
| Production format | **`tool_result` block** | The structured message that carries a tool's output back to the model, tied to the request by its `id`. | Replaces the `TOOL RESULT: ...` message of the text loop. |
| Production format | **`AgentTurn` / `hit_cap`** | The outcome of one question: the answer, the steps taken, and whether the iteration cap stopped the loop. | The section 4 comparison maps the final return to `AgentTurn(answer, steps, hit_cap)`. |
| Tool choice | **Tool description** | The text that tells the model what a tool covers, when to use it, and when to use another tool instead. | `CURRENT_DATE_NARROW` limits `current_date` to questions about today's date. |
| Tool choice | **Behavior case** | A question paired with the tool that should answer it, used to check tool choice after every change. | `("Which documents do you have?", "list_sources")` in `CHOICE_CASES`. |
| Tool choice | **Description collision** | Two tools whose descriptions both claim the same questions, so the model can pick the wrong one. | `current_date`'s description claimed *how many days until*, so it beat `days_until`. |
| Tool choice | **Stand-in model** | A scripted or rule-based replacement for a real model, used to test the loop offline and on purpose. | `scripted(...)` controls section 4 turns; `description_reader` checks tool choice in sections 5–6. |
| Hardening | **Read-only vs. action tool** | A read-only tool only looks things up; an action tool changes something (writes a file, saves memory). | `search_docs` is read-only; `save_note`, `save_draft` and `remember` have `read_only=False`. |
| Hardening | **Approval gate** | An execution check that requires approval for the exact action before it runs. | Review the destination and content of a proposed draft; refusal prevents the write. |
| Hardening | **Prompt injection** | An attempt by untrusted content to redirect the agent's behavior. | A retrieved password-reset document tells the agent to save private information. |
| Hardening | **Least privilege** | Give a tool access only to the data and operations it needs. | A draft tool can write only to its designated folder. |
| Hardening | **Bounded recovery** | Limit attempts to recover from a failure, then stop or ask for help. | Retry a temporary service failure a limited number of times; check whether an action happened before retrying it. |
| Hardening | **Timeout** | A limit on how long a call may take. | Stop waiting for an unavailable service even if the loop has iterations left. |
| Hardening | **Grounding / score floor** | Grounding ties claims to evidence; a score floor filters retrieval matches below a threshold. | Check whether a cited policy supports the answer; abstain if evidence is insufficient. |
| Hardening | **Trace** | A record of calls and outcomes used to investigate behavior. | Spot repeated failed calls while keeping keys and private content out of the log. |
| Hardening | **Rotate a key** | Replace an exposed key and invalidate the old one at the provider. | A key committed to a repository needs revocation or rotation even after the file is removed. |
| Interoperability | **MCP (Model Context Protocol)** | A protocol through which servers can expose tools for agents to discover and call. | Tools from an MCP server join the available menu and still need validation and permission checks. |
